# Phase 5b: Linear and Regularized Models

This notebook estimates OLS, Ridge, and Lasso prediction models for `unmet_need_pc` using the Phase 5a modeling dataset and time-aware splits.

## Setup and Data

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'modeling_dataset_5a_with_splits.csv'
FEATURE_PATH = PROJECT_ROOT / 'outputs' / 'model_feature_set_5a.md'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
feature_lines = FEATURE_PATH.read_text(encoding='utf-8').splitlines()
features = [line.strip()[3:-1] for line in feature_lines if line.strip().startswith('- `') and line.strip().endswith('`')]

outcome = 'unmet_need_pc'
missing_features = [feature for feature in features if feature not in df.columns]
if missing_features:
    raise ValueError(f'Missing features in modeling dataset: {missing_features}')

X = df[features]
y = df[outcome]

X_train = df.loc[df['split'] == 'train', features]
y_train = df.loc[df['split'] == 'train', outcome]
X_valid = df.loc[df['split'] == 'valid', features]
y_valid = df.loc[df['split'] == 'valid', outcome]
X_test = df.loc[df['split'] == 'test', features]
y_test = df.loc[df['split'] == 'test', outcome]

pd.DataFrame({
    'split': ['train', 'valid', 'test'],
    'rows': [len(X_train), len(X_valid), len(X_test)],
    'min_year': [df.loc[df['split'] == split, 'year'].min() for split in ['train', 'valid', 'test']],
    'max_year': [df.loc[df['split'] == split, 'year'].max() for split in ['train', 'valid', 'test']],
})

,split,rows,min_year,max_year
0,train,102,2015,2018
1,valid,52,2019,2020
2,test,76,2021,2023


## Standardization

The scaler is fitted on the training rows only. The same fitted scaler is then applied to validation and test rows.

In [2]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

X_train_valid = pd.concat([X_train, X_valid], axis=0)
y_train_valid = pd.concat([y_train, y_valid], axis=0)
scaler_train_valid = StandardScaler()
X_train_valid_scaled = scaler_train_valid.fit_transform(X_train_valid)
X_test_scaled_train_valid = scaler_train_valid.transform(X_test)

In [3]:
def evaluate_predictions(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred),
    }


def coefficient_table(model, feature_names):
    return pd.DataFrame({
        'feature': feature_names,
        'coefficient': model.coef_.ravel(),
    }).sort_values('coefficient', key=lambda s: s.abs(), ascending=False).reset_index(drop=True)

## OLS Model

In [4]:
ols_model = LinearRegression()
ols_model.fit(X_train_scaled, y_train)

ols_valid_pred = ols_model.predict(X_valid_scaled)
ols_test_pred = ols_model.predict(X_test_scaled)

ols_valid_metrics = evaluate_predictions(y_valid, ols_valid_pred)
ols_test_metrics = evaluate_predictions(y_test, ols_test_pred)
ols_valid_metrics, ols_test_metrics

({'mae': 2.6211074424241048,
  'rmse': np.float64(3.286165872118118),
  'r2': -0.3337048858984295},
 {'mae': 1.9377258977200056,
  'rmse': np.float64(2.4036408170539802),
  'r2': 0.002759530829517076})

## Ridge and Lasso Tuning

In [5]:
alpha_grid = [0.01, 0.1, 1.0, 10.0, 100.0]

def tune_regularized_model(model_class, model_name, alpha_values):
    rows = []
    for alpha in alpha_values:
        if model_name == 'Lasso':
            model = model_class(alpha=alpha, max_iter=100000)
        else:
            model = model_class(alpha=alpha)
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_valid_scaled)
        metrics = evaluate_predictions(y_valid, pred)
        rows.append({
            'model_name': model_name,
            'alpha': alpha,
            'mae_valid': metrics['mae'],
            'rmse_valid': metrics['rmse'],
            'r2_valid': metrics['r2'],
        })
    tuning = pd.DataFrame(rows).sort_values(['rmse_valid', 'mae_valid', 'alpha']).reset_index(drop=True)
    best_alpha = float(tuning.loc[0, 'alpha'])
    if model_name == 'Lasso':
        final_model = model_class(alpha=best_alpha, max_iter=100000)
    else:
        final_model = model_class(alpha=best_alpha)
    final_model.fit(X_train_valid_scaled, y_train_valid)
    test_pred = final_model.predict(X_test_scaled_train_valid)
    test_metrics = evaluate_predictions(y_test, test_pred)
    return tuning, best_alpha, final_model, test_metrics

ridge_tuning, ridge_alpha, ridge_model, ridge_test_metrics = tune_regularized_model(Ridge, 'Ridge', alpha_grid)
lasso_tuning, lasso_alpha, lasso_model, lasso_test_metrics = tune_regularized_model(Lasso, 'Lasso', alpha_grid)

ridge_tuning, lasso_tuning

(  model_name   alpha  mae_valid  rmse_valid  r2_valid
 0      Ridge  100.00   1.613327    2.534297  0.206776
 1      Ridge   10.00   2.230526    2.904031 -0.041558
 2      Ridge    1.00   2.521649    3.144758 -0.221392
 3      Ridge    0.10   2.591050    3.232346 -0.290376
 4      Ridge    0.01   2.617266    3.279200 -0.328056,
   model_name   alpha  mae_valid  rmse_valid  r2_valid
 0      Lasso    0.10   2.079196    2.810610  0.024377
 1      Lasso    1.00   1.950734    2.831989  0.009479
 2      Lasso   10.00   2.031297    2.898374 -0.037504
 3      Lasso  100.00   2.031297    2.898374 -0.037504
 4      Lasso    0.01   2.519654    3.148594 -0.224374)

## Performance and Coefficients

In [6]:
performance = pd.DataFrame([
    {
        'model_name': 'OLS',
        'alpha': np.nan,
        'mae_test': ols_test_metrics['mae'],
        'rmse_test': ols_test_metrics['rmse'],
        'r2_test': ols_test_metrics['r2'],
    },
    {
        'model_name': 'Ridge',
        'alpha': ridge_alpha,
        'mae_test': ridge_test_metrics['mae'],
        'rmse_test': ridge_test_metrics['rmse'],
        'r2_test': ridge_test_metrics['r2'],
    },
    {
        'model_name': 'Lasso',
        'alpha': lasso_alpha,
        'mae_test': lasso_test_metrics['mae'],
        'rmse_test': lasso_test_metrics['rmse'],
        'r2_test': lasso_test_metrics['r2'],
    },
])
performance.to_csv(OUTPUTS_DIR / 'table_linear_models_performance.csv', index=False)

coefficient_table(ols_model, features).to_csv(OUTPUTS_DIR / 'coefficients_ols.csv', index=False)
coefficient_table(ridge_model, features).to_csv(OUTPUTS_DIR / 'coefficients_ridge.csv', index=False)
coefficient_table(lasso_model, features).to_csv(OUTPUTS_DIR / 'coefficients_lasso.csv', index=False)

ridge_tuning.to_csv(OUTPUTS_DIR / 'ridge_validation_tuning.csv', index=False)
lasso_tuning.to_csv(OUTPUTS_DIR / 'lasso_validation_tuning.csv', index=False)

performance

,model_name,alpha,mae_test,rmse_test,r2_test
0,OLS,NaN,1.937726,2.403641,0.002760
1,Ridge,100.0,1.373545,1.945226,0.346868
2,Lasso,0.1,1.428230,1.887122,0.385303


In [7]:
best = performance.sort_values(['rmse_test', 'mae_test']).iloc[0]
ols = performance.loc[performance['model_name'] == 'OLS'].iloc[0]
regularized = performance.loc[performance['model_name'].isin(['Ridge', 'Lasso'])].sort_values(['rmse_test', 'mae_test']).iloc[0]

if regularized['rmse_test'] < ols['rmse_test']:
    reg_sentence = (
        f"Among the regularized models, {regularized['model_name']} has the lower test RMSE. "
        f"It improves on OLS by {ols['rmse_test'] - regularized['rmse_test']:.3f} RMSE points and "
        f"{ols['mae_test'] - regularized['mae_test']:.3f} MAE points."
    )
else:
    reg_sentence = (
        f"Regularization does not improve test RMSE relative to OLS in this split. "
        f"The best regularized model is {regularized['model_name']}, with test RMSE "
        f"{regularized['rmse_test']:.3f} compared with {ols['rmse_test']:.3f} for OLS."
    )

summary_parts = [
    '# Linear and regularized models summary',
    '',
    (
        f"The best test-set result among OLS, Ridge, and Lasso is from {best['model_name']}, "
        f"with MAE {best['mae_test']:.3f}, RMSE {best['rmse_test']:.3f}, and R-squared {best['r2_test']:.3f}. "
        'The comparison uses the Phase 5a time-aware split and the same selected feature set for all three models.'
    ),
    '',
    reg_sentence,
    '',
    'The coefficients describe predictive associations in aggregate Eurostat country-year data. They should be read as model summaries for prediction, not as causal claims.',
    '',
]
summary = '\n'.join(summary_parts)
(OUTPUTS_DIR / 'linear_models_summary.md').write_text(summary, encoding='utf-8')
print(summary)

# Linear and regularized models summary

The best test-set result among OLS, Ridge, and Lasso is from Lasso, with MAE 1.428, RMSE 1.887, and R-squared 0.385. The comparison uses the Phase 5a time-aware split and the same selected feature set for all three models.

Among the regularized models, Lasso has the lower test RMSE. It improves on OLS by 0.517 RMSE points and 0.509 MAE points.

The coefficients describe predictive associations in aggregate Eurostat country-year data. They should be read as model summaries for prediction, not as causal claims.

